# İlaç Etkileşim Modeli — Değerlendirme (Colab)

Bu notebook, `magahcicek/mistral7b-drug-interaction-lora` modelini **60 örneklik**
test seti üzerinde çalıştırır ve basit bir **doğruluk oranı** üretir.

Kaynak: TİTCK KÜB §4.5. **Çalıştırma sırası:** Runtime → Change runtime type → **GPU (T4)**, sonra tüm hücreleri sırayla çalıştırın.

> ⚠️ Yalnızca araştırma/eğitim amaçlıdır; klinik karar için kullanılamaz.


In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate

In [ ]:
# Hugging Face girişi (token gizli olarak istenir; Mistral/Trendyol bazı modeller için gerekir)
from getpass import getpass
from huggingface_hub import login
login(getpass("HF token (hf_...): "))

In [ ]:
%%writefile grading.py
# -*- coding: utf-8 -*-
"""Serbest metin model çıktılarını etiketlere çeviren ve metrik hesaplayan modül.

Saf Python; ağır bağımlılık (torch/transformers) gerektirmez, böylece GPU
olmadan da test edilebilir. Model çıktısı serbest Türkçe metin olduğundan,
"etkileşim var/yok" kararı basit bir anahtar-kelime sezgiseliyle çıkarılır.
Bu, önerilen "basit doğruluk oranı" yaklaşımıdır.
"""

from __future__ import annotations

import re
import unicodedata
from typing import Dict, List, Optional

# Çıktıda "risk/etkileşim var" sinyali veren ifadeler.
# NOT: Yalın "etkilesim" kelimesi RİSK sinyali DEĞİLDİR; çünkü olumsuz
# cümlelerde de geçer ("anlamlı etkileşim beklenmez"). Onun yerine açık
# risk/yan-etki kökleri ve "etkileşim var/mevcut" gibi olumlu kalıplar aranır.
RISK_INDICATORS = [
    "risk", "artar", "artabilir", "artir", "artis", "yukselt", "yuksel",
    "tehlike", "kanama", "yan etki", "advers", "zararl", "onerilmez",
    "kacinil", "dikkat", "olumsuz", "komplikasyon", "toksik", "hiperkalemi",
    "bronkospazm", "aritmi", "solunum", "baskil", "serotonin sendromu",
    "nefrotoksik", "hepatotoksi", "kotules", "bozab", "bozar", "boz ",
    "azalt", "sedasyon", "hasar", "zorlas", "tetikle",
    "yol acab", "neden olab", "neden olur",
    "etkilesime", "ciddi etkilesim", "onemli etkilesim", "etkilesim var",
    "etkilesim mevcut", "etkilesim soz konusu",
]

# Çıktıda "etkileşim yok / güvenli" sinyali veren ifadeler
SAFE_INDICATORS = [
    "etkilesim yok", "etkilesim bulunmamak", "anlamli etkilesim yok",
    "anlamli bir etkilesim", "guvenli", "guvenle", "sorun olusturmaz",
    "birlikte kullanilabil", "risk tasimaz", "zararsiz", "beklenmez",
    "sakinca yok", "sakincasi yok",
]


def _normalize(text: str) -> str:
    """Türkçe karakterleri sadeleştirip küçük harfe çevirir (eşleştirme için)."""
    text = text.casefold()
    # ı/İ ve diğer aksanları ASCII'ye indir
    replacements = {"ı": "i", "ş": "s", "ğ": "g", "ü": "u", "ö": "o", "ç": "c", "â": "a", "î": "i", "û": "u"}
    for src, dst in replacements.items():
        text = text.replace(src, dst)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", text)


def predict_interaction(output_text: str) -> Optional[bool]:
    """Model çıktısından 'etkileşim var mı?' kararını çıkarır.

    Dönüş:
        True  -> çıktı bir etkileşim/risk tarif ediyor
        False -> çıktı güvenli/etkileşim yok diyor
        None  -> karar verilemiyor (boş veya belirsiz çıktı)
    """
    if not output_text or not output_text.strip():
        return None

    norm = _normalize(output_text)
    has_safe = any(ind in norm for ind in SAFE_INDICATORS)
    has_risk = any(ind in norm for ind in RISK_INDICATORS)

    if has_safe and not has_risk:
        return False
    if has_risk:
        return True
    if has_safe:
        return False
    return None


def mechanism_recall(output_text: str, expected_keywords: List[str]) -> Optional[float]:
    """Beklenen mekanizma anahtar kelimelerinin çıktıda bulunma oranı.

    Beklenen kelime yoksa None döner (recall tanımsız).
    """
    if not expected_keywords:
        return None
    norm = _normalize(output_text or "")
    hits = sum(1 for kw in expected_keywords if _normalize(kw) in norm)
    return hits / len(expected_keywords)


def compute_metrics(records: List[Dict]) -> Dict:
    """Tahmin edilmiş kayıtlardan metrikleri hesaplar.

    Her kayıt: {"etkilesim_var": bool, "tahmin": Optional[bool],
                "mekanizma": [...], "cikti": str}
    """
    total = len(records)
    scored = [r for r in records if r.get("tahmin") is not None]
    abstained = total - len(scored)

    tp = sum(1 for r in scored if r["etkilesim_var"] and r["tahmin"])
    tn = sum(1 for r in scored if not r["etkilesim_var"] and not r["tahmin"])
    fp = sum(1 for r in scored if not r["etkilesim_var"] and r["tahmin"])
    fn = sum(1 for r in scored if r["etkilesim_var"] and not r["tahmin"])

    correct = tp + tn
    accuracy = correct / len(scored) if scored else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    # Mekanizma kapsamı (yalnızca pozitif örnekler)
    recalls = [
        mechanism_recall(r.get("cikti", ""), r.get("mekanizma", []))
        for r in records if r["etkilesim_var"]
    ]
    recalls = [x for x in recalls if x is not None]
    mech_coverage = sum(recalls) / len(recalls) if recalls else 0.0

    return {
        "toplam": total,
        "puanlanan": len(scored),
        "cekimser": abstained,
        "dogru": correct,
        "dogruluk": accuracy,
        "kesinlik": precision,
        "duyarlilik": recall,
        "f1": f1,
        "mekanizma_kapsami": mech_coverage,
        "karisiklik_matrisi": {"TP": tp, "TN": tn, "FP": fp, "FN": fn},
    }


def format_report(metrics: Dict, model_id: str = "?") -> str:
    """Metrikleri okunabilir bir Markdown raporuna çevirir."""
    cm = metrics["karisiklik_matrisi"]
    lines = [
        "# Değerlendirme Raporu",
        "",
        f"- **Model:** `{model_id}`",
        f"- **Test seti boyutu:** {metrics['toplam']} örnek "
        f"(puanlanan: {metrics['puanlanan']}, çekimser: {metrics['cekimser']})",
        f"- **Kaynak:** TİTCK KÜB §4.5 (klasik, belgelenmiş etkileşimler)",
        "",
        "## Etkileşim Tespiti (var/yok)",
        "",
        "| Metrik | Değer |",
        "|--------|-------|",
        f"| Doğruluk (accuracy) | **{metrics['dogruluk']:.1%}** |",
        f"| Kesinlik (precision) | {metrics['kesinlik']:.1%} |",
        f"| Duyarlılık (recall) | {metrics['duyarlilik']:.1%} |",
        f"| F1 | {metrics['f1']:.1%} |",
        f"| Mekanizma kapsamı | {metrics['mekanizma_kapsami']:.1%} |",
        "",
        "## Karışıklık Matrisi",
        "",
        "| | Tahmin: Var | Tahmin: Yok |",
        "|---|---|---|",
        f"| **Gerçek: Var** | {cm['TP']} (TP) | {cm['FN']} (FN) |",
        f"| **Gerçek: Yok** | {cm['FP']} (FP) | {cm['TN']} (TN) |",
        "",
        "> Not: Etkileşim var/yok kararı, serbest metin çıktıdan anahtar-kelime",
        "> sezgiseliyle çıkarılır. Bu, basit ve şeffaf bir doğruluk ölçütüdür;",
        "> klinik geçerlilik iddiası taşımaz.",
    ]
    return "\n".join(lines)


In [ ]:
import json
TEST_SET = json.loads(r'''[
 {
  "id": "pos-001",
  "girdi_1": "warfarin",
  "girdi_1_tip": "ilac",
  "girdi_2": "aspirin",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "kanama",
   "antiagregan",
   "antikoagülan"
  ],
  "referans_ozet": "Warfarin ile aspirin birlikte kullanımı kanama riskini belirgin şekilde artırır.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-002",
  "girdi_1": "warfarin",
  "girdi_1_tip": "ilac",
  "girdi_2": "ağrı kesici",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "kanama",
   "NSAİİ",
   "gastrointestinal"
  ],
  "referans_ozet": "Warfarin ile NSAİİ türü ağrı kesiciler gastrointestinal kanama riskini artırır.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-003",
  "girdi_1": "warfarin",
  "girdi_1_tip": "ilac",
  "girdi_2": "iltihap kurutucu",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "kanama",
   "NSAİİ"
  ],
  "referans_ozet": "Warfarin ile NSAİİ (iltihap kurutucu) birlikte kullanımı kanamaya yol açabilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-004",
  "girdi_1": "warfarin",
  "girdi_1_tip": "ilac",
  "girdi_2": "antibiyotik",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "INR",
   "kanama"
  ],
  "referans_ozet": "Bazı antibiyotikler warfarinin INR değerini yükselterek kanama riskini artırır.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-005",
  "girdi_1": "warfarin",
  "girdi_1_tip": "ilac",
  "girdi_2": "parol",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "INR",
   "parasetamol"
  ],
  "referans_ozet": "Yüksek doz parasetamol warfarinin INR değerini yükseltebilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-006",
  "girdi_1": "klopidogrel",
  "girdi_1_tip": "ilac",
  "girdi_2": "aspirin",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "kanama",
   "antiagregan"
  ],
  "referans_ozet": "Klopidogrel ve aspirin (ikili antiagregan) kanama riskini artırır.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-007",
  "girdi_1": "klopidogrel",
  "girdi_1_tip": "ilac",
  "girdi_2": "mide koruyucu",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "omeprazol",
   "etkinlik",
   "CYP2C19"
  ],
  "referans_ozet": "Omeprazol gibi mide koruyucular klopidogrelin antiagregan etkinliğini azaltabilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-008",
  "girdi_1": "aspirin",
  "girdi_1_tip": "ilac",
  "girdi_2": "ağrı kesici",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "gastrointestinal",
   "ülser",
   "kanama"
  ],
  "referans_ozet": "Aspirin ile diğer NSAİİ ağrı kesiciler ülser ve GİS kanama riskini artırır.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-009",
  "girdi_1": "aspirin",
  "girdi_1_tip": "ilac",
  "girdi_2": "iltihap kurutucu",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "gastrointestinal",
   "kanama"
  ],
  "referans_ozet": "Aspirin ile NSAİİ birlikte kullanımı gastrointestinal kanamayı artırır.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-010",
  "girdi_1": "aspirin",
  "girdi_1_tip": "ilac",
  "girdi_2": "kan sulandırıcı",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "kanama"
  ],
  "referans_ozet": "Aspirin ile kan sulandırıcı birlikte kullanımı kanama riskini artırır.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-011",
  "girdi_1": "kan sulandırıcı",
  "girdi_1_tip": "ilac",
  "girdi_2": "ağrı kesici",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "kanama",
   "NSAİİ"
  ],
  "referans_ozet": "Kan sulandırıcı ile NSAİİ ağrı kesici kanama riskini ciddi şekilde artırır.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-012",
  "girdi_1": "kan sulandırıcı",
  "girdi_1_tip": "ilac",
  "girdi_2": "aspirin",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "kanama"
  ],
  "referans_ozet": "Kan sulandırıcı ile aspirin birlikte kullanımı kanamaya yol açabilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-013",
  "girdi_1": "miğren",
  "girdi_1_tip": "ilac",
  "girdi_2": "antidepresan",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "serotonin sendromu",
   "triptan",
   "SSRI"
  ],
  "referans_ozet": "Triptan (miğren ilacı) ile SSRI antidepresan serotonin sendromuna yol açabilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-014",
  "girdi_1": "antidepresan",
  "girdi_1_tip": "ilac",
  "girdi_2": "sakinleştirici",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "sedasyon",
   "MSS"
  ],
  "referans_ozet": "Antidepresan ile sakinleştirici birlikte kullanımı merkezi sinir sistemi depresyonunu artırır.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-015",
  "girdi_1": "antidepresan",
  "girdi_1_tip": "ilac",
  "girdi_2": "sinir ilacı",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "serotonin",
   "MSS"
  ],
  "referans_ozet": "Antidepresan ile sinir ilacı serotonerjik ve MSS yan etkilerini artırabilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-016",
  "girdi_1": "potasyum",
  "girdi_1_tip": "ilac",
  "girdi_2": "idrar söktürücü",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "hiperkalemi",
   "potasyum"
  ],
  "referans_ozet": "Potasyum tutucu idrar söktürücü ile potasyum hiperkalemiye yol açabilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-017",
  "girdi_1": "potasyum takviyesi",
  "girdi_1_tip": "ilac",
  "girdi_2": "idrar söktürücü",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "hiperkalemi",
   "potasyum"
  ],
  "referans_ozet": "Potasyum takviyesi ile potasyum tutucu diüretik hiperkalemi riskini artırır.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-018",
  "girdi_1": "barbitürat",
  "girdi_1_tip": "ilac",
  "girdi_2": "sakinleştirici",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "solunum depresyonu",
   "MSS"
  ],
  "referans_ozet": "Barbitürat ile sakinleştirici birlikte solunum depresyonuna yol açabilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-019",
  "girdi_1": "barbitürat",
  "girdi_1_tip": "ilac",
  "girdi_2": "antidepresan",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "MSS",
   "enzim"
  ],
  "referans_ozet": "Barbitürat ile antidepresan MSS depresyonu ve enzim indüksiyonuna neden olabilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-020",
  "girdi_1": "anestezi",
  "girdi_1_tip": "ilac",
  "girdi_2": "sakinleştirici",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "solunum depresyonu",
   "MSS"
  ],
  "referans_ozet": "Anestezik ile sakinleştirici additif MSS ve solunum depresyonu yapar.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-021",
  "girdi_1": "kas gevşetici",
  "girdi_1_tip": "ilac",
  "girdi_2": "sakinleştirici",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "sedasyon",
   "solunum"
  ],
  "referans_ozet": "Kas gevşetici ile sakinleştirici additif sedasyon ve solunum baskılanması yapar.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-022",
  "girdi_1": "kas gevşetici",
  "girdi_1_tip": "ilac",
  "girdi_2": "barbitürat",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "MSS"
  ],
  "referans_ozet": "Kas gevşetici ile barbitürat merkezi sinir sistemi depresyonunu artırır.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-023",
  "girdi_1": "adrenalin",
  "girdi_1_tip": "ilac",
  "girdi_2": "antidepresan",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "hipertansif",
   "kan basıncı"
  ],
  "referans_ozet": "Adrenalin ile bazı antidepresanlar (MAOI/TCA) hipertansif yanıta yol açabilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-024",
  "girdi_1": "parol",
  "girdi_1_tip": "ilac",
  "girdi_2": "ateş düşürücü",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "parasetamol",
   "doz",
   "karaciğer"
  ],
  "referans_ozet": "Parol ile ateş düşürücü çoğunlukla parasetamol içerir; doz aşımı karaciğer hasarı yapabilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-025",
  "girdi_1": "metformin",
  "girdi_1_tip": "ilac",
  "girdi_2": "idrar söktürücü",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "kan şekeri",
   "glisemik"
  ],
  "referans_ozet": "İdrar söktürücüler glisemik kontrolü bozarak metformin etkisini etkileyebilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-026",
  "girdi_1": "antibiyotik",
  "girdi_1_tip": "ilac",
  "girdi_2": "hormon",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "kontraseptif",
   "etkinlik"
  ],
  "referans_ozet": "Bazı antibiyotikler oral kontraseptif (hormon) etkinliğini azaltabilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-027",
  "girdi_1": "hipertansiyon",
  "girdi_1_tip": "hastalik",
  "girdi_2": "grip ilacı",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "kan basıncı",
   "psödoefedrin",
   "dekonjestan"
  ],
  "referans_ozet": "Hipertansiyon hastalarında psödoefedrin içeren grip ilaçları kan basıncını yükseltir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-028",
  "girdi_1": "tansiyon",
  "girdi_1_tip": "hastalik",
  "girdi_2": "grip ilacı",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "kan basıncı",
   "psödoefedrin"
  ],
  "referans_ozet": "Tansiyon hastalarında dekonjestan içeren grip ilaçları kan basıncını yükseltir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-029",
  "girdi_1": "hipertansiyon",
  "girdi_1_tip": "hastalik",
  "girdi_2": "ağrı kesici",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "kan basıncı",
   "NSAİİ",
   "sıvı"
  ],
  "referans_ozet": "NSAİİ ağrı kesiciler sıvı tutulumu yaparak kan basıncını yükseltir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-030",
  "girdi_1": "tansiyon",
  "girdi_1_tip": "hastalik",
  "girdi_2": "ağrı kesici",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "kan basıncı",
   "NSAİİ"
  ],
  "referans_ozet": "NSAİİ ağrı kesiciler tansiyon kontrolünü zorlaştırabilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-031",
  "girdi_1": "hipertansiyon",
  "girdi_1_tip": "hastalik",
  "girdi_2": "iltihap kurutucu",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "kan basıncı",
   "NSAİİ"
  ],
  "referans_ozet": "NSAİİ (iltihap kurutucu) hipertansiyonda kan basıncını yükseltebilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-032",
  "girdi_1": "tansiyon",
  "girdi_1_tip": "hastalik",
  "girdi_2": "potasyum",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "hiperkalemi",
   "potasyum"
  ],
  "referans_ozet": "ACE inhibitörü kullanan tansiyon hastalarında potasyum hiperkalemiye yol açar.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-033",
  "girdi_1": "tansiyon",
  "girdi_1_tip": "hastalik",
  "girdi_2": "potasyum takviyesi",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "hiperkalemi",
   "potasyum"
  ],
  "referans_ozet": "Tansiyon ilaçları ile potasyum takviyesi hiperkalemi riskini artırır.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-034",
  "girdi_1": "kalp",
  "girdi_1_tip": "hastalik",
  "girdi_2": "grip ilacı",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "aritmi",
   "kan basıncı"
  ],
  "referans_ozet": "Kalp hastalarında dekonjestan içeren grip ilaçları aritmi ve kan basıncı artışı yapabilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-035",
  "girdi_1": "böbrek",
  "girdi_1_tip": "hastalik",
  "girdi_2": "ağrı kesici",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "böbrek",
   "nefrotoksik",
   "NSAİİ"
  ],
  "referans_ozet": "NSAİİ ağrı kesiciler böbrek hastalarında nefrotoksiktir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-036",
  "girdi_1": "böbrek",
  "girdi_1_tip": "hastalik",
  "girdi_2": "iltihap kurutucu",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "böbrek",
   "NSAİİ"
  ],
  "referans_ozet": "NSAİİ (iltihap kurutucu) böbrek fonksiyonlarını kötüleştirebilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-037",
  "girdi_1": "astım",
  "girdi_1_tip": "hastalik",
  "girdi_2": "aspirin",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "bronkospazm",
   "astım"
  ],
  "referans_ozet": "Aspirin duyarlı astımlılarda aspirin bronkospazma yol açabilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-038",
  "girdi_1": "astım",
  "girdi_1_tip": "hastalik",
  "girdi_2": "ağrı kesici",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "bronkospazm",
   "NSAİİ"
  ],
  "referans_ozet": "NSAİİ ağrı kesiciler duyarlı astımlılarda bronkospazm yapabilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-039",
  "girdi_1": "karaciğer",
  "girdi_1_tip": "hastalik",
  "girdi_2": "parol",
  "etkilesim_var": true,
  "siddet": "major",
  "mekanizma": [
   "karaciğer",
   "parasetamol",
   "hepatotoksik"
  ],
  "referans_ozet": "Karaciğer hastalarında parasetamol (parol) hepatotoksisite riskini artırır.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "pos-040",
  "girdi_1": "gut",
  "girdi_1_tip": "hastalik",
  "girdi_2": "idrar söktürücü",
  "etkilesim_var": true,
  "siddet": "orta",
  "mekanizma": [
   "ürik asit",
   "gut"
  ],
  "referans_ozet": "İdrar söktürücüler ürik asidi yükselterek gut ataklarını tetikleyebilir.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-001",
  "girdi_1": "aspirin",
  "girdi_1_tip": "ilac",
  "girdi_2": "parol",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [
   "parasetamol"
  ],
  "referans_ozet": "Aspirin ve parasetamol (parol) birlikte yaygın kullanılır; anlamlı etkileşim beklenmez.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-002",
  "girdi_1": "parol",
  "girdi_1_tip": "ilac",
  "girdi_2": "antibiyotik",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [],
  "referans_ozet": "Parasetamol ile çoğu antibiyotik arasında klinik olarak anlamlı etkileşim beklenmez.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-003",
  "girdi_1": "metformin",
  "girdi_1_tip": "ilac",
  "girdi_2": "parol",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [],
  "referans_ozet": "Metformin ile parasetamol arasında anlamlı bir ilaç-ilaç etkileşimi beklenmez.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-004",
  "girdi_1": "parol",
  "girdi_1_tip": "ilac",
  "girdi_2": "mide koruyucu",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [],
  "referans_ozet": "Parasetamol ile mide koruyucular arasında anlamlı etkileşim beklenmez.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-005",
  "girdi_1": "ateş düşürücü",
  "girdi_1_tip": "ilac",
  "girdi_2": "mide koruyucu",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [],
  "referans_ozet": "Parasetamol türü ateş düşürücü ile mide koruyucu arasında anlamlı etkileşim beklenmez.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-006",
  "girdi_1": "alerji",
  "girdi_1_tip": "ilac",
  "girdi_2": "parol",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [],
  "referans_ozet": "Alerji ilacı (antihistaminik) ile parasetamol arasında anlamlı etkileşim beklenmez.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-007",
  "girdi_1": "nezle",
  "girdi_1_tip": "hastalik",
  "girdi_2": "parol",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [],
  "referans_ozet": "Nezle durumunda parasetamol güvenle kullanılır; anlamlı etkileşim beklenmez.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-008",
  "girdi_1": "grip",
  "girdi_1_tip": "hastalik",
  "girdi_2": "parol",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [],
  "referans_ozet": "Grip durumunda parasetamol güvenle kullanılır; anlamlı etkileşim beklenmez.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-009",
  "girdi_1": "diyabet",
  "girdi_1_tip": "hastalik",
  "girdi_2": "parol",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [],
  "referans_ozet": "Diyabet hastalarında parasetamol ile anlamlı bir etkileşim beklenmez.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-010",
  "girdi_1": "hipertansiyon",
  "girdi_1_tip": "hastalik",
  "girdi_2": "parol",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [],
  "referans_ozet": "Hipertansiyonda parasetamol tercih edilen analjeziktir; anlamlı etkileşim beklenmez.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-011",
  "girdi_1": "tansiyon",
  "girdi_1_tip": "hastalik",
  "girdi_2": "parol",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [],
  "referans_ozet": "Tansiyon hastalarında parasetamol genellikle güvenlidir; anlamlı etkileşim beklenmez.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-012",
  "girdi_1": "kansızlık",
  "girdi_1_tip": "hastalik",
  "girdi_2": "parol",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [],
  "referans_ozet": "Kansızlık (anemi) durumunda parasetamol ile anlamlı etkileşim beklenmez.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-013",
  "girdi_1": "obezite",
  "girdi_1_tip": "hastalik",
  "girdi_2": "parol",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [],
  "referans_ozet": "Obezitede parasetamol ile anlamlı bir ilaç etkileşimi beklenmez.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-014",
  "girdi_1": "astım",
  "girdi_1_tip": "hastalik",
  "girdi_2": "parol",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [],
  "referans_ozet": "Astımda parasetamol NSAİİ'ye tercih edilir; anlamlı etkileşim beklenmez.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-015",
  "girdi_1": "zatürre",
  "girdi_1_tip": "hastalik",
  "girdi_2": "parol",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [],
  "referans_ozet": "Zatürre durumunda parasetamol ile anlamlı bir etkileşim beklenmez.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-016",
  "girdi_1": "romatizma",
  "girdi_1_tip": "hastalik",
  "girdi_2": "mide koruyucu",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [],
  "referans_ozet": "Romatizma hastalarında mide koruyucu NSAİİ ile birlikte güvenle kullanılır.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-017",
  "girdi_1": "nezle",
  "girdi_1_tip": "hastalik",
  "girdi_2": "antibiyotik",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [],
  "referans_ozet": "Nezlede antibiyotik ile parasetamol/destek tedavisi arasında anlamlı etkileşim beklenmez.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-018",
  "girdi_1": "grip",
  "girdi_1_tip": "hastalik",
  "girdi_2": "mide koruyucu",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [],
  "referans_ozet": "Grip durumunda mide koruyucu ile anlamlı bir etkileşim beklenmez.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-019",
  "girdi_1": "zatürre",
  "girdi_1_tip": "hastalik",
  "girdi_2": "mide koruyucu",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [],
  "referans_ozet": "Zatürre durumunda mide koruyucu ile anlamlı bir etkileşim beklenmez.",
  "kaynak": "TİTCK KÜB §4.5"
 },
 {
  "id": "neg-020",
  "girdi_1": "kalp",
  "girdi_1_tip": "hastalik",
  "girdi_2": "parol",
  "etkilesim_var": false,
  "siddet": "yok",
  "mekanizma": [],
  "referans_ozet": "Kalp hastalarında parasetamol genellikle güvenli analjeziktir; anlamlı etkileşim beklenmez.",
  "kaynak": "TİTCK KÜB §4.5"
 }
]''')
print('Test seti:', len(TEST_SET), 'ornek')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig

PEFT_ID = "magahcicek/mistral7b-drug-interaction-lora"
cfg = PeftConfig.from_pretrained(PEFT_ID)
print("Temel model:", cfg.base_model_name_or_path)

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(cfg.base_model_name_or_path)
base = AutoModelForCausalLM.from_pretrained(
    cfg.base_model_name_or_path, quantization_config=bnb, device_map="auto")
model = PeftModel.from_pretrained(base, PEFT_ID)
model.eval()
print("Model hazir.")

In [ ]:
def build_prompt(ex):
    q = "İlaç etkileşimlerini değerlendirerek, potansiyel sağlık risklerini açıklayın.\n"
    if ex["girdi_1_tip"] == "ilac":
        return f"[INST] {q} {ex['girdi_1']} ve {ex['girdi_2']} birlikte kullanımı hangi kronik hastalıklara neden olur? \n[/INST]"
    return f"[INST] {q} {ex['girdi_1']} hastasıyım ve {ex['girdi_2']} birlikte kullanımı hangi kronik hastalıklara neden olur? \n[/INST]"

@torch.no_grad()
def generate(ex, max_new_tokens=256):
    p = build_prompt(ex)
    batch = tok(p, return_tensors="pt").to(model.device)
    gen = model.generate(**batch, max_new_tokens=max_new_tokens, eos_token_id=tok.eos_token_id)
    new = gen[0][batch["input_ids"].shape[1]:]
    return tok.decode(new, skip_special_tokens=True).strip()

In [ ]:
from grading import predict_interaction, compute_metrics, format_report

LIMIT = 0  # 0 = tum test seti; hizli deneme icin orn. 10
rows = TEST_SET[:LIMIT] if LIMIT else TEST_SET

results = []
for i, ex in enumerate(rows, 1):
    txt = generate(ex)
    results.append({**ex, "cikti": txt, "tahmin": predict_interaction(txt)})
    print(f"[{i}/{len(rows)}] {ex['girdi_1']} + {ex['girdi_2']} -> tahmin={results[-1]['tahmin']}")

m = compute_metrics(results)
report = format_report(m, model_id="magahcicek/mistral7b-drug-interaction-lora")
print("\n" + report)
print(f"\n>>> DOGRULUK: {m['dogruluk']:.1%} ({m['dogru']}/{m['puanlanan']})")

with open("report.md", "w", encoding="utf-8") as f:
    f.write(report + "\n")
with open("predictions.jsonl", "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("\nKaydedildi: report.md, predictions.jsonl  (Files panelinden indirebilirsiniz)")